In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
train_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/bank_customer_churn/train.csv')

# Display the first few rows of the dataset
print(train_data.head())

# Check the shape of the dataset
print(train_data.shape)

# Check for missing values
print(train_data.isnull().sum())

# Check data types
print(train_data.dtypes)

# Separate numerical and categorical columns
numerical_cols = train_data.select_dtypes(include=[np.number]).columns
categorical_cols = train_data.select_dtypes(include=['object', 'category']).columns

# Display summary statistics for numerical columns
print(train_data[numerical_cols].describe())

# Display value counts for categorical columns
for col in categorical_cols:
    print(train_data[col].value_counts())

# Visualize missing values
plt.figure(figsize=(10, 6))
sns.heatmap(train_data.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

# Visualize the distribution of numerical columns
train_data[numerical_cols].hist(bins=20, figsize=(15, 10))
plt.suptitle('Distribution of Numerical Features')
plt.show()

# Visualize the distribution of categorical columns
for col in categorical_cols:
    plt.figure(figsize=(10, 6))
    sns.countplot(y=train_data[col])
    plt.title(f'Distribution of {col}')
    plt.show()

# Visualize correlation matrix for numerical columns
plt.figure(figsize=(12, 8))
correlation_matrix = train_data[numerical_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()


       id  CustomerId     Surname  ...  IsActiveMember EstimatedSalary Exited
0  149380    15780088  Yobachukwu  ...             1.0       103560.98      0
1  164766    15679760    Slattery  ...             0.0       102950.79      0
2  155569    15637678          Ma  ...             0.0       155394.52      0
3  124304    15728693      Galkin  ...             1.0       107428.42      0
4  108008    15613673        Lung  ...             0.0       134110.93      0

[5 rows x 14 columns]
(132027, 14)
id                 0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64
id                   int64
CustomerId           int64
Surname             object
CreditScore          int64
Geography           object
Gender              object
Age                float64


In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_data)
print("column_info")
print(column_info)


2025-09-14 22:52:19.776 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': ['Surname', 'Geography', 'Gender'], 'Numeric': ['id', 'CustomerId', 'CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, LabelEncode, StandardScale

# Load the training data
train_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/bank_customer_churn/train.csv')

# Separate categorical and numerical columns
categorical_cols = ['Surname', 'Geography', 'Gender']
numerical_cols = ['id', 'CustomerId', 'CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']

# Handle missing values
fill_missing = FillMissingValue(features=numerical_cols + categorical_cols, strategy='most_frequent')
train_data = fill_missing.fit_transform(train_data)

# Encode categorical variables
label_encode = LabelEncode(features=categorical_cols)
train_data = label_encode.fit_transform(train_data)

# Normalize numerical features
standard_scale = StandardScale(features=numerical_cols)
train_data = standard_scale.fit_transform(train_data)

# Display the preprocessed data
print(train_data.head())
print(train_data.dtypes)


         id  CustomerId  Surname  ...  IsActiveMember  EstimatedSalary  Exited
0  1.403375    1.236497     2703  ...        1.005415        -0.181454       0
1  1.725899   -0.169062     2302  ...       -0.994614        -0.193591       0
2  1.533110   -0.758615     1510  ...       -0.994614         0.849538       0
3  0.877727    0.516472      888  ...        1.005415        -0.104529       0
4  0.536127   -1.094916     1505  ...       -0.994614         0.426198       0

[5 rows x 14 columns]
id                 float64
CustomerId         float64
Surname              int32
CreditScore        float64
Geography            int32
Gender               int32
Age                float64
Tenure             float64
Balance            float64
NumOfProducts      float64
HasCrCard          float64
IsActiveMember     float64
EstimatedSalary    float64
Exited               int64
dtype: object


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_data)
print("column_info")
print(column_info)


column_info
{'Category': [], 'Numeric': ['id', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited'], 'Datetime': [], 'Others': []}


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# Load the preprocessed data
train_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/bank_customer_churn/train.csv')

# Define features and target
X = train_data.drop(columns=['Exited'])
y = train_data['Exited']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the XGBoost model
model = XGBClassifier(
    objective='binary:logistic',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(X_train, y_train)

# Predict probabilities on the test set
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Calculate the AUC-ROC score
auc_roc = roc_auc_score(y_test, y_pred_proba)
print(f'AUC-ROC Score: {auc_roc:.4f}')


ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:Surname: object, Geography: object, Gender: object

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# Load the preprocessed data
train_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/bank_customer_churn/train.csv')

# Convert categorical columns to 'category' type
categorical_cols = ['Surname', 'Geography', 'Gender']
train_data[categorical_cols] = train_data[categorical_cols].astype('category')

# Define features and target
X = train_data.drop(columns=['Exited'])
y = train_data['Exited']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the XGBoost model
model = XGBClassifier(
    objective='binary:logistic',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(X_train, y_train)

# Predict probabilities on the test set
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Calculate the AUC-ROC score
auc_roc = roc_auc_score(y_test, y_pred_proba)
print(f'AUC-ROC Score: {auc_roc:.4f}')


ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:Surname: category, Geography: category, Gender: category